Explore date types, partition keys and date predicates.


In [ ]:
from setup.nbs.inspect_tools import show, show_methods
from pair_recon_left.common.sql.dates import DateType, NativeDateType, StringDateType, NumericDateType, SasDateType
from pair_recon_left.common.sql.dates import resolve_date_type, canonical_date, bucket_date, partition_key
from pair_recon_left.common.sql.dates import aggregate_partition_counts, partition_where, bounds_where

show(resolve_date_type, canonical_date, bucket_date, partition_key, aggregate_partition_counts)
show(partition_where, bounds_where)
show_methods(DateType, 'detect', 'predicate', 'normalize', 'partition_key', 'normalize_sql', 'spec')
show_methods(NativeDateType, 'day', 'normalize_sql')
show_methods(StringDateType, 'detect', 'normalize', 'normalize_sql', 'spec')
show_methods(NumericDateType, 'normalize', 'normalize_sql', 'partition_key', 'spec')
show_methods(SasDateType, 'detect', 'normalize', 'normalize_sql', 'spec')


In [ ]:
from pair_recon_left.common.sql.dates import resolve_date_type

# 同一个接口覆盖 date、compact string、numeric YYYYMM 和 SAS datetime
for spec in ('date', {'type': 'string', 'format': 'compact'}, {'type': 'number', 'format': 'month'}, {'type': 'sas', 'format': 'datetime'}):
    print(spec, '->', resolve_date_type(spec).spec(portable=True))


## Date and partition helpers

These examples show concrete date values and generated SQL; no database is contacted.


In [ ]:
print("canonical_date:", "2026-01-02", "->", canonical_date("2026-01-02"))
print("partition_key:", "2026-01-02", "month ->", partition_key("2026-01-02", "month"))
print("partition_where:", partition_where("ORDER_DT", ["2026-01-02"], dialect="oracle", date_type="date", grain="day"))
print("bounds_where:", bounds_where("ORDER_DT", dialect="oracle", date_type="date", from_date="2026-01-01", to_date="2026-01-31"))


## Date type methods

Each object receives a representative value and emits its portable specification, normalized value, partition key and SQL. Detection resolves `auto` formats from concrete values.


In [ ]:
examples = [
    (NativeDateType(), "2026-01-02"),
    (StringDateType(format="compact"), "20260102"),
    (NumericDateType(format="day"), 20260102),
    (SasDateType(format="days"), 24108),
]
for date_type, value in examples:
    print("\nclass:", type(date_type).__name__, "input:", value)
    print("spec:", date_type.spec(portable=True))
    print("normalize:", date_type.normalize(value))
    try:
        print("partition_key:", date_type.partition_key(value))
    except ValueError as error:
        print("partition_key error:", str(error))
    print("predicate:", date_type.predicate("ORDER_DT", "oracle"))
    print("normalize_sql:", date_type.normalize_sql("ORDER_DT", "oracle"))
    print("day:", date_type.day("ORDER_DT", "oracle"))
for date_type, values in [(StringDateType(), ["20260102"]), (SasDateType(), [24108])]:
    print("detect:", type(date_type).__name__, values, "->", date_type.detect(values).spec(portable=True))


## Toy example

Run this cell locally to inspect the concrete inputs and outputs.


In [ ]:
counts = {"2026-01-01": 2, "2026-01-02": 3, "2026-02-01": 4}
print("input:", counts)
print("monthly:", aggregate_partition_counts(counts, "month"))
print("bucket:", bucket_date("2026-01-02", "month"))
